# Session 5: The Agentic Paradigm and Tool Calling

Until now the LLM could only **talk**: answer from its training (Session 2) or from documents we handed it (Session 4).
Today it starts to **act**: look up live data, calculate exactly, change things in a system.

The mechanism is **tool calling**, and an **agent** is just a loop around it. We build that loop from scratch first, so nothing about agents stays magic.

| Part | What we do |
|---|---|
| 0 | Setup and a tiny "bank system" to act on |
| 1 | What LLMs cannot do alone |
| 2 | One tool call by hand, with the raw SDK |
| 3 | The agent loop from scratch (ReAct) |
| 4 | Parallel calls and tool errors |
| 5 | LangChain: `@tool`, `bind_tools`, `ToolMessage` |
| 6 | `create_agent`: the prebuilt loop |
| 7 | Designing good tools (and dangerous ones) |
| 8 | RAG as a tool: the agent decides when to search |
| 9 | Workflow or agent? Cost of autonomy |
| 10 | Exercises |

## Setup: run this cell first

Choose the provider and model in the cell below. The API key is **not** written here: it stays in the `.env` file in this folder, for example:

```
OPENROUTER_API_KEY=sk-or-v1-...
```

In [1]:
# ===================== SETUP: run this cell first =====================
PROVIDER = "openrouter"    # "openrouter", "gemini" or "openai"
MODEL = "gpt-4o-mini"                 # leave empty to use the default model below

import os
from dotenv import load_dotenv

# read the .env file in this folder (override=True picks up edits without restarting the kernel)
"""if not os.path.exists():
    raise FileNotFoundError("No .env file in this folder. Create it and add your API key.")""";
load_dotenv(override=True)

if PROVIDER == "openrouter":
    KEY_NAME = "OPENROUTER_API_KEY"
    MODEL = MODEL or "google/gemini-2.5-flash-lite"
    BASE_URL = "https://openrouter.ai/api/v1"
elif PROVIDER == "gemini":
    KEY_NAME = "GOOGLE_API_KEY"
    MODEL = MODEL or "gemini-2.5-flash-lite"
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
elif PROVIDER == "openai":
    KEY_NAME = "OPENAI_API_KEY"
    MODEL = MODEL or "gpt-4.1-mini"
    BASE_URL = "https://api.openai.com/v1"
else:
    raise ValueError('PROVIDER must be "openrouter", "gemini" or "openai"')

API_KEY = os.getenv(KEY_NAME)
if not API_KEY:
    raise ValueError(f"{KEY_NAME} is missing. Add it to the .env file in this folder.")


def get_chat_model(temperature=0, **kwargs):
    """LangChain chat model for the chosen provider."""
    if PROVIDER == "openrouter":
        from langchain_openrouter import ChatOpenRouter
        return ChatOpenRouter(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)
    elif PROVIDER == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODEL, api_key=API_KEY, temperature=temperature, **kwargs)
    else:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)


def get_openai_client():
    """Plain OpenAI SDK client pointed at the chosen provider. Returns (client, model)."""
    from openai import OpenAI
    return OpenAI(api_key=API_KEY, base_url=BASE_URL), MODEL


def show_config():
    """Print the settings without showing the full key."""
    print(f"provider : {PROVIDER}")
    print(f"model    : {MODEL}")
    print(f"key      : {KEY_NAME} = {API_KEY[:6]}...{API_KEY[-2:]} ({len(API_KEY)} chars)")
    print(f"endpoint : {BASE_URL}")


show_config()

provider : openrouter
model    : gpt-4o-mini
key      : OPENROUTER_API_KEY = sk-or-...1d (73 chars)
endpoint : https://openrouter.ai/api/v1


## Part 0: Setup

In [2]:
import os
import re
import json
import glob
import math
from datetime import date
from importlib.metadata import version

for pkg in ["openai", "langchain", "langchain-core", "langgraph"]:
    print(f"{pkg:15s} {version(pkg)}")

show_config()

client, MODEL = get_openai_client()
llm = get_chat_model(temperature=0)

openai          3.26.1
langchain       1.4.3
langchain-core  1.6.7
langgraph       1.2.14
provider : openrouter
model    : gpt-4o-mini
key      : OPENROUTER_API_KEY = sk-or-...1d (73 chars)
endpoint : https://openrouter.ai/api/v1


### A tiny bank system

Agents act on real systems: databases, APIs, ticketing tools. We fake one with a Python dict, so we can see exactly what the agent reads and changes. Customer `C1001` is the logged-in user in our demos.

In [3]:
CUSTOMERS = {
    "C1001": {"name": "Riya Sharma", "balance": 48250.75, "card_last4": "4417", "card_status": "active"},
    "C1002": {"name": "Arjun Patil", "balance": 1520.00, "card_last4": "9021", "card_status": "active"},
}

for cid, c in CUSTOMERS.items():
    print(cid, c)

C1001 {'name': 'Riya Sharma', 'balance': 48250.75, 'card_last4': '4417', 'card_status': 'active'}
C1002 {'name': 'Arjun Patil', 'balance': 1520.0, 'card_last4': '9021', 'card_status': 'active'}


## Part 1: What LLMs cannot do alone

Three kinds of questions an LLM alone gets wrong:

In [4]:
questions = [
    "What is today's date?",
    "What is my account balance? I am customer C1001.",
    "What is the monthly EMI on a loan of Rs 3,00,000 at 11.75% per year for 24 months? Give the exact rupee amount.",
]
for q in questions:
    print("Q:", q)
    print("A:", llm.invoke(q).content)
    print()

Q: What is today's date?
A: Today's date is October 4, 2023.

Q: What is my account balance? I am customer C1001.
A: I'm sorry, but I don't have access to personal account information or databases. To check your account balance, please contact your bank or service provider directly through their official website or customer service.

Q: What is the monthly EMI on a loan of Rs 3,00,000 at 11.75% per year for 24 months? Give the exact rupee amount.
A: To calculate the monthly EMI (Equated Monthly Installment) for a loan, we can use the formula:

\[
EMI = \frac{P \times r \times (1 + r)^n}{(1 + r)^n - 1}
\]

Where:
- \( P \) = Principal loan amount (Rs 3,00,000)
- \( r \) = Monthly interest rate (annual rate / 12)
- \( n \) = Number of monthly installments (loan tenure in months)

Given:
- Principal \( P = 3,00,000 \)
- Annual interest rate = 11.75% = 0.1175
- Monthly interest rate \( r = \frac{0.1175}{12} = 0.00979167 \)
- Loan tenure \( n = 24 \) months

Now, substituting the values int

- **Live or private data** (today's date, a customer's balance): not in the training data. The model refuses or invents.
- **Exact computation**: LLMs predict text; multi-step arithmetic is often slightly wrong, and slightly wrong is unacceptable for money.
- **Actions** (block a card, create a ticket): the model can only produce text.

The fix: give the model **tools** - normal Python functions - and let it ask us to run them.

**Analogy:** a manager who cannot touch the computer but knows exactly what to ask for. "Please look up the balance for C1001", "please run the EMI calculator with these numbers". An assistant (our code) does it and reports back. The manager decides; the assistant executes.

## Part 2: One tool call, by hand

### Step 1: ordinary Python functions

Tools are plain functions. Nothing AI about them. Note they return **strings**: the result goes back into the conversation as text.

In [5]:
def get_today_date():
    return date.today().isoformat()


def get_account_balance(customer_id):
    customer = CUSTOMERS.get(customer_id)
    if customer is None:
        return f"ERROR: no customer with id {customer_id}. Ask the user for a valid customer id."
    return f"Balance for {customer_id}: Rs {customer['balance']:,.2f}"


def calculate_emi(principal, annual_rate_percent, months):
    r = annual_rate_percent / 12 / 100
    emi = principal * r * (1 + r) ** months / ((1 + r) ** months - 1)
    total = emi * months
    return f"EMI: Rs {emi:,.2f} per month. Total payable: Rs {total:,.2f}. Total interest: Rs {total - principal:,.2f}."


print(get_today_date())
print(get_account_balance("C1001"))
print(calculate_emi(300000, 11.75, 24))

2026-11-01
Balance for C1001: Rs 48,250.75
EMI: Rs 14,087.04 per month. Total payable: Rs 338,089.02. Total interest: Rs 38,089.02.


### Step 2: describe the tools to the model

The model never sees our Python code. It sees a **JSON schema**: name, description, and the arguments with types. This description is a prompt - the model decides **whether and how** to call a tool by reading it.

In [6]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_today_date",
            "description": "Get today's date in YYYY-MM-DD format.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_account_balance",
            "description": "Get the current account balance of a bank customer.",
            "parameters": {
                "type": "object",
                "properties": {"customer_id": {"type": "string", "description": "Customer id, for example C1001"}},
                "required": ["customer_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_emi",
            "description": "Calculate the exact monthly EMI, total payable and total interest for a loan.",
            "parameters": {
                "type": "object",
                "properties": {
                    "principal": {"type": "number", "description": "Loan amount in rupees"},
                    "annual_rate_percent": {"type": "number", "description": "Yearly interest rate in percent, e.g. 11.75"},
                    "months": {"type": "integer", "description": "Loan tenure in months"},
                },
                "required": ["principal", "annual_rate_percent", "months"],
            },
        },
    },
]

TOOL_FUNCTIONS = {
    "get_today_date": get_today_date,
    "get_account_balance": get_account_balance,
    "calculate_emi": calculate_emi,
}

### Step 3: send the question together with the tools

In [7]:
messages = [
    {"role": "system", "content": "You are a banking assistant. Use tools for dates, balances and calculations."},
    {"role": "user", "content": "What is the EMI on Rs 3,00,000 at 11.75% for 24 months?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
msg = response.choices[0].message

print("finish_reason:", response.choices[0].finish_reason)
print("content      :", msg.content)
print("tool_calls   :")
for call in msg.tool_calls or []:
    print("   id       :", call.id)
    print("   name     :", call.function.name)
    print("   arguments:", call.function.arguments, "<- a JSON STRING")

finish_reason: tool_calls
content      : None
tool_calls   :
   id       : call_iUaOJuL9SV57yiTRN8qKeo9W
   name     : calculate_emi
   arguments: {"principal":300000,"annual_rate_percent":11.75,"months":24} <- a JSON STRING


Read this carefully:

- `finish_reason` is `tool_calls`, and `content` is empty: the model did **not** answer. It asked us to run a function.
- The model **did not run anything**. It only produced a name and JSON arguments. Running code is always our job - which is also where we keep control.
- `arguments` is a JSON **string**; we parse it.

### Step 4: run the tool and send the result back

Two messages go back: the assistant's tool-call message (so the model remembers what it asked), and a `tool` message with the result, linked by `tool_call_id`.

In [8]:
messages.append(msg.model_dump(exclude_none=True))   # keep the assistant's request exactly as returned

for call in msg.tool_calls:
    args = json.loads(call.function.arguments)
    result = TOOL_FUNCTIONS[call.function.name](**args)
    print(f"running {call.function.name}({args}) -> {result}")
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
print("\nfinish_reason:", final.choices[0].finish_reason)
print("answer:", final.choices[0].message.content)

running calculate_emi({'principal': 300000, 'annual_rate_percent': 11.75, 'months': 24}) -> EMI: Rs 14,087.04 per month. Total payable: Rs 338,089.02. Total interest: Rs 38,089.02.

finish_reason: stop
answer: The EMI on Rs 3,00,000 at an interest rate of 11.75% for 24 months is as follows:

- **EMI**: Rs 14,087.04 per month
- **Total Payable**: Rs 3,38,089.02
- **Total Interest**: Rs 38,089.02


In [9]:
final

ChatCompletion(id='gen-1793507923-lesfaWIGPEyvTPYG8U5e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The EMI on Rs 3,00,000 at an interest rate of 11.75% for 24 months is as follows:\n\n- **EMI**: Rs 14,087.04 per month\n- **Total Payable**: Rs 3,38,089.02\n- **Total Interest**: Rs 38,089.02', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1793507923, model='openai/gpt-4o-mini', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_e9e83bb9dc', usage=CompletionUsage(completion_tokens=75, prompt_tokens=254, total_tokens=329, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_wri

The whole conversation the model saw. Four messages: system, user, assistant (tool request), tool (result) - then it wrote the answer.

In [10]:
for m in messages:
    role = m["role"]
    text = m.get("content") or json.dumps([c["function"] for c in m.get("tool_calls", [])])
    print(f"{role:9s} | {str(text)[:110]}")

system    | You are a banking assistant. Use tools for dates, balances and calculations.
user      | What is the EMI on Rs 3,00,000 at 11.75% for 24 months?
assistant | [{"arguments": "{\"principal\":300000,\"annual_rate_percent\":11.75,\"months\":24}", "name": "calculate_emi"}]
tool      | EMI: Rs 14,087.04 per month. Total payable: Rs 338,089.02. Total interest: Rs 38,089.02.


## Part 3: The agent loop, from scratch

Some questions need several tool calls in sequence, decided as we go. We cannot hard-code the number of steps. So we **loop**: call the model; if it asks for tools, run them and loop again; if it answers, stop.

This loop is called **ReAct** (Reason + Act): the model reasons about what to do, acts (asks for a tool), observes the result, and repeats.

```
question -> model -> tool calls? --yes--> run tools -> add results -> model -> ...
                                 --no---> final answer
```

A `max_steps` limit is essential: without it a confused model can loop forever and spend money.

In [11]:
SYSTEM = (
    "You are a banking assistant for Sahyadri Digital Bank. The logged-in customer is C1001. "
    "Use tools for dates, balances and calculations; never guess numbers. Be brief."
)

def run_agent(question, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] final answer")
            return msg.content

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            try:
                result = TOOL_FUNCTIONS[call.function.name](**args)
            except Exception as e:
                result = f"ERROR: {type(e).__name__}: {e}"
            if verbose:
                print(f"[step {step}] {call.function.name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

    return "Stopped: reached the maximum number of steps without an answer."

In [12]:
print(run_agent("What is my balance?"))
print()
print(run_agent("What is today's date?"))

[step 1] get_account_balance({'customer_id': 'C1001'}) -> Balance for C1001: Rs 48,250.75
[step 2] final answer
Your current balance is Rs 48,250.75.

[step 1] get_today_date({}) -> 2026-11-01
[step 2] final answer
Today's date is 2026-11-01.


Notice that `C1001` came from the **system prompt**, not from the user. Passing context like the logged-in user's id in the system prompt (or better, binding it in code - Part 7) is how real assistants know who they serve.

A question that needs reasoning across steps:

In [14]:
print(run_agent(
    "I want to borrow Rs 3,00,000 for 24 months at 11.75%. "
    "Calculate the EMI, then tell me whether my current balance covers at least 6 EMIs."
    "also check today's date and what will be my last date of repayment?"
))

[step 1] calculate_emi({'principal': 300000, 'annual_rate_percent': 11.75, 'months': 24}) -> EMI: Rs 14,087.04 per month. Total payable: Rs 338,089.02. Total interest: Rs 38,089.02.
[step 1] get_account_balance({'customer_id': 'C1001'}) -> Balance for C1001: Rs 48,250.75
[step 1] get_today_date({}) -> 2026-11-01
[step 2] final answer
Your EMI for the loan of Rs 3,00,000 at 11.75% for 24 months is Rs 14,087.04.

Your current balance is Rs 48,250.75, which does not cover at least 6 EMIs (Rs 84,522.24).

Today's date is 2026-11-01, making your last date of repayment 2028-10-31.


## Part 4: Parallel calls and tool errors

### Parallel tool calls

When the steps are independent, models can request **several tools in one turn**. Our loop already handles it: it runs every call in `msg.tool_calls`.

In [15]:
print(run_agent("Tell me today's date and my account balance."))

[step 1] get_today_date({}) -> 2026-11-01
[step 1] get_account_balance({'customer_id': 'C1001'}) -> Balance for C1001: Rs 48,250.75
[step 2] final answer
Today's date is **2026-11-01** and your account balance is **Rs 48,250.75**.


If both tools were requested in the same step, you see two lines with the same `[step 1]`. One round trip instead of two: faster and cheaper.

### Tool errors are information, not crashes

Our loop catches exceptions and sends the error **text** back to the model. A good error message tells the model what to do next.

In [21]:
SYSTEM = (
    "You are a banking assistant for Sahyadri Digital Bank. respond based on the logged in customer"
    "at a time only one customer can be logged in"
    "Use tools for dates, balances and calculations; never guess numbers. Be brief."
)

def run_agent(question, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS_SCHEMA)
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] final answer")
            return msg.content

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            try:
                result = TOOL_FUNCTIONS[call.function.name](**args)
            except Exception as e:
                result = f"ERROR: {type(e).__name__}: {e}"
            if verbose:
                print(f"[step {step}] {call.function.name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

    return "Stopped: reached the maximum number of steps without an answer."

In [22]:
print(get_account_balance("C9999"))
print()
print(run_agent("What is the balance of customer C9999 and customer C1001?"))

ERROR: no customer with id C9999. Ask the user for a valid customer id.

[step 1] final answer
I can only check the balance for one logged-in customer at a time. Please let me know which customer's balance you would like to check: C9999 or C1001?


The model reads `ERROR: no customer with id C9999. Ask the user for a valid customer id.` and responds to the user sensibly, instead of the program crashing. Write tool errors **for the model**: what went wrong and what to do.

## Part 5: The LangChain way: `@tool` and `bind_tools`

Writing JSON schemas by hand is tedious and error-prone. LangChain builds them from the function itself:

- **name** = function name
- **description** = the docstring
- **arguments** = the type hints (and `Annotated` / `Literal` / Pydantic for details)

In [23]:
# decorator 
def shout(text):
    return text.upper()

f = shout   # not calling it just a reference

print(f('hi'))

HI


In [24]:
def logger(func):

    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__} with {args} {kwargs}")

        result = func(*args, **kwargs)

        print(f"{func.__name__} returned {result}")

        return result

    return wrapper

def add(a, b):
    return a + b

# manual decorator
add = logger(add)

add(2, 5)


calling add with (2, 5) {}
add returned 7


7

In [26]:
@logger
def add(a, b):
    return a + b

In [27]:
add(3 ,6)

calling add with (3, 6) {}
add returned 9


9

In [28]:
"""  
tool decorator ->

it builds the tool schema => name, what the tool does, what argument it takes and what type

""";

In [29]:
from typing import Annotated, Literal
from langchain_core.tools import tool


@tool
def get_today_date_tool() -> str:
    """Get today's date in YYYY-MM-DD format."""
    return date.today().isoformat()


@tool
def get_account_balance_tool(customer_id: Annotated[str, "Customer id, for example C1001"]) -> str:
    """Get the current account balance of a bank customer."""
    return get_account_balance(customer_id)


@tool
def calculate_emi_tool(
    principal: Annotated[float, "Loan amount in rupees"],
    annual_rate_percent: Annotated[float, "Yearly interest rate in percent, e.g. 11.75"],
    months: Annotated[int, "Loan tenure in months"],
) -> str:
    """Calculate the exact monthly EMI, total payable and total interest for a loan."""
    return calculate_emi(principal, annual_rate_percent, months)


print(type(calculate_emi_tool).__name__)
print("name       :", calculate_emi_tool.name)
print("description:", calculate_emi_tool.description)
print("args       :", calculate_emi_tool.args)

StructuredTool
name       : calculate_emi_tool
description: Calculate the exact monthly EMI, total payable and total interest for a loan.
args       : {'principal': {'description': 'Loan amount in rupees', 'title': 'Principal', 'type': 'number'}, 'annual_rate_percent': {'description': 'Yearly interest rate in percent, e.g. 11.75', 'title': 'Annual Rate Percent', 'type': 'number'}, 'months': {'description': 'Loan tenure in months', 'title': 'Months', 'type': 'integer'}}


Under the hood: the schema LangChain sends is the same JSON we wrote by hand in Part 2.

In [30]:
from langchain_core.utils.function_calling import convert_to_openai_tool
print(json.dumps(convert_to_openai_tool(calculate_emi_tool), indent=2))

{
  "type": "function",
  "function": {
    "name": "calculate_emi_tool",
    "description": "Calculate the exact monthly EMI, total payable and total interest for a loan.",
    "parameters": {
      "properties": {
        "principal": {
          "description": "Loan amount in rupees",
          "type": "number"
        },
        "annual_rate_percent": {
          "description": "Yearly interest rate in percent, e.g. 11.75",
          "type": "number"
        },
        "months": {
          "description": "Loan tenure in months",
          "type": "integer"
        }
      },
      "required": [
        "principal",
        "annual_rate_percent",
        "months"
      ],
      "type": "object"
    }
  }
}


A tool is a Runnable, so you can test it directly - **always test tools without the LLM first**:

In [31]:
print(calculate_emi_tool.invoke({"principal": 300000, "annual_rate_percent": 11.75, "months": 24}))

EMI: Rs 14,087.04 per month. Total payable: Rs 338,089.02. Total interest: Rs 38,089.02.


`llm.bind_tools(tools)` attaches the schemas to every call. The reply's `tool_calls` are already parsed into dicts (no `json.loads`), in the same format for every provider.

In [32]:
lc_tools = [get_today_date_tool, get_account_balance_tool, calculate_emi_tool]
llm_with_tools = llm.bind_tools(lc_tools)

ai_msg = llm_with_tools.invoke("What is the EMI on Rs 5,00,000 at 10.5% for 36 months?")
print("content   :", repr(ai_msg.content))
print("tool_calls:", ai_msg.tool_calls)

content   : ''
tool_calls: [{'name': 'calculate_emi_tool', 'args': {'principal': 500000, 'annual_rate_percent': 10.5, 'months': 36}, 'id': 'call_cSUYiThDAaWRvZ7cCtZdlOtN', 'type': 'tool_call'}]


The same loop as Part 3, with LangChain messages. `ToolMessage` is the `tool` role message.

In [33]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

tools_by_name = {t.name: t for t in lc_tools}

def run_lc_agent(question, max_steps=6):
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    for step in range(1, max_steps + 1):
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.content
        for call in ai.tool_calls:
            result = tools_by_name[call["name"]].invoke(call["args"])
            print(f"[step {step}] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "Stopped: too many steps."

print(run_lc_agent("What is the EMI on Rs 5,00,000 at 10.5% for 36 months?"))

[step 1] calculate_emi_tool({'principal': 500000, 'annual_rate_percent': 10.5, 'months': 36}) -> EMI: Rs 16,251.22 per month. Total payable: Rs 585,043.98. Total interest: Rs 85,043.98.
The EMI on Rs 5,00,000 at an interest rate of 10.5% for 36 months is Rs 16,251.22 per month. The total payable amount will be Rs 585,043.98, with a total interest of Rs 85,043.98.


## Part 6: `create_agent`: the prebuilt loop

The loop we wrote twice is so common that LangChain ships it: `create_agent`. It is built on **LangGraph** (Session 7), so it also supports memory, streaming, human approval and more.

**Library-change trap:** older tutorials use `AgentExecutor`, `initialize_agent` or `create_react_agent`. In LangChain 1.x the standard is `from langchain.agents import create_agent`.

In [34]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=lc_tools,
    system_prompt=SYSTEM,
)

result = agent.invoke({"messages": [{"role": "user", "content": "What is my balance and today's date?"}]})

print("final answer:", result["messages"][-1].content)

final answer: Your account balance is Rs 48,250.75, and today's date is 2026-11-01.


The result holds the **full message history**: the same human, AI (tool calls), tool and final AI messages our own loop produced. Inspecting it is how you debug an agent.

In [35]:
for m in result["messages"]:
    kind = type(m).__name__
    if getattr(m, "tool_calls", None):
        detail = "calls " + ", ".join(f"{c['name']}({c['args']})" for c in m.tool_calls)
    else:
        detail = str(m.content)[:100]
    print(f"{kind:12s} | {detail}")

HumanMessage | What is my balance and today's date?
AIMessage    | calls get_account_balance_tool({'customer_id': 'C1001'}), get_today_date_tool({})
ToolMessage  | Balance for C1001: Rs 48,250.75
ToolMessage  | 2026-11-01
AIMessage    | Your account balance is Rs 48,250.75, and today's date is 2026-11-01.


**Streaming the steps.** `stream_mode="updates"` yields each step as it happens - what a UI shows as "Looking up your balance...".

In [36]:
for update in agent.stream(
    {"messages": [{"role": "user", "content": "Calculate the EMI for Rs 2,00,000 at 13.5% for 12 months."}]},
    stream_mode="updates",
):
    for node, data in update.items():
        last = data["messages"][-1]
        calls = getattr(last, "tool_calls", None)
        print(f"[{node}]", f"tool calls: {[c['name'] for c in calls]}" if calls else str(last.content)[:120])

[model] tool calls: ['calculate_emi_tool']
[tools] EMI: Rs 17,910.41 per month. Total payable: Rs 214,924.87. Total interest: Rs 14,924.87.
[model] The EMI for a loan of Rs 2,00,000 at an interest rate of 13.5% for 12 months is Rs 17,910.41. 

- Total payable: Rs 2,14


## Part 7: Designing good tools (and dangerous ones)

The model chooses tools **only from their names, descriptions and argument schemas**. Tool design is prompt design.

### A vague tool vs a clear tool

Both functions do the same thing. Watch which one the model can actually use.

In [41]:
@tool
def upper(x: str) -> str:
    """Helper function."""
    return x.upper()


@tool
def get_customer_balance(customer_id: Annotated[str, "Customer id like C1001"]) -> str:
    """Get the current savings account balance in rupees for a customer id.
    Use this whenever the user asks how much money they have."""
    return get_account_balance(customer_id)


for tools in [[upper], [get_customer_balance]]:
    bound = llm.bind_tools(tools)
    ai = bound.invoke([SystemMessage(SYSTEM), HumanMessage("How much money do I have as C1001?")])
    print(tools[0].name, "->", ai.tool_calls or ai.content[:100])

upper -> [{'name': 'upper', 'args': {'x': 'C1001'}, 'id': 'call_jy56fS4e9Oi7tm7CMlGjWLZe', 'type': 'tool_call'}]
get_customer_balance -> [{'name': 'get_customer_balance', 'args': {'customer_id': 'C1001'}, 'id': 'call_3qjYS9easV76CPWWXmXHDbLB', 'type': 'tool_call'}]


**Rules for good tools**

| Rule | Why |
|---|---|
| Specific name (`get_customer_balance`, not `helper`) | the name is the first thing the model reads |
| Docstring says what it does **and when to use it** | the model matches the user's need to this text |
| Typed, described arguments; `Literal` for fixed choices | fewer invalid arguments |
| Return short, clear text | results go into the context; long results cost tokens and confuse |
| Errors say what to do next | the model can recover |
| One job per tool | easier to choose correctly and to test |

### Tools that change things

Reading a balance is harmless. **Blocking a card** is an action with consequences. Two protections:

1. **Do not let the model choose who it acts on.** The customer id should come from the logged-in session (code), not from the model's arguments - otherwise a clever prompt could act on someone else's account.
2. **Confirm before acting.** Here: a `confirm` flag. In Session 7, LangGraph pauses the agent for real human approval.

In [42]:
LOGGED_IN_CUSTOMER = "C1001"   # comes from the app's login, never from the model


@tool
def block_card(
    reason: Literal["lost", "stolen", "damaged"],
    confirm: Annotated[bool, "Must be true only after the user has explicitly confirmed blocking"] = False,
) -> str:
    """Permanently block the logged-in customer's debit card. Ask the user to confirm first.
    Call with confirm=false to get the confirmation question, confirm=true only after the user says yes."""
    customer = CUSTOMERS[LOGGED_IN_CUSTOMER]
    if not confirm:
        return (f"CONFIRMATION NEEDED: blocking card ending {customer['card_last4']} is permanent. "
                "Ask the user to reply yes to confirm.")
    customer["card_status"] = f"blocked ({reason})"
    return f"Card ending {customer['card_last4']} is now blocked. A replacement can be requested in the app."


print(block_card.invoke({"reason": "lost"}))
print(CUSTOMERS[LOGGED_IN_CUSTOMER]["card_status"])

CONFIRMATION NEEDED: blocking card ending 4417 is permanent. Ask the user to reply yes to confirm.
active


In [43]:
card_agent = create_agent(model=llm, tools=[block_card, get_customer_balance], system_prompt=SYSTEM)

out = card_agent.invoke({"messages": [{"role": "user", "content": "I lost my card, please block it."}]})
print(out["messages"][-1].content)
print("card status now:", CUSTOMERS[LOGGED_IN_CUSTOMER]["card_status"])

Please confirm that you want to block your debit card due to it being lost. Do you want to proceed?
card status now: active


The first turn should end with a confirmation question, and the card should still be `active`. A follow-up "yes" message would confirm; Session 7 shows how the agent remembers the conversation and pauses for approval properly.

**Never** rely only on the prompt for safety: the `confirm` flag and the code-bound customer id are enforced in Python, whatever the model says.

## Part 8: RAG as a tool

In Session 4 every question went through retrieval. An agent can **decide** when to search the policy documents, when to calculate, and when to do both.

First we rebuild the Session 4 index (compact version of the same steps).

In [44]:
import chromadb
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
NOISE = [r"^Sahyadri Digital Bank \|.*$", r"^Sahyadri Digital Bank Ltd\. \|.*$", r"^Page \d+$"]
STATUS = {"Personal_Loan_Policy_2024.pdf": "superseded"}

pages = []
for path in sorted(glob.glob("docs/*.pdf")):
    name = os.path.basename(path)
    for i, page in enumerate(PdfReader(path).pages):
        lines = [l for l in (page.extract_text() or "").splitlines() if not any(re.match(p, l.strip()) for p in NOISE)]
        pages.append(Document("\n".join(lines), metadata={"source": name, "page": i + 1, "status": STATUS.get(name, "current")}))

chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120, add_start_index=True).split_documents(pages)

db = chromadb.PersistentClient(path="chroma_db")
if "bank_policies_s5" in [c.name for c in db.list_collections()]:
    db.delete_collection("bank_policies_s5")
store = Chroma(client=db, collection_name="bank_policies_s5",
               embedding_function=HuggingFaceEmbeddings(model=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True}),
               collection_configuration={"hnsw": {"space": "cosine"}})
store.add_documents(chunks, ids=[f"{c.metadata['source']}-p{c.metadata['page']}-{c.metadata['start_index']}" for c in chunks])
policy_retriever = store.as_retriever(search_kwargs={"k": 3, "filter": {"status": "current"}})
print(len(chunks), "chunks indexed")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

15 chunks indexed


Now wrap the retriever as a tool. The docstring tells the model **when** to use it. The result includes sources, so the agent can cite them.

In [45]:
@tool
def search_bank_policies(query: Annotated[str, "What to look up, in plain words"]) -> str:
    """Search Sahyadri Digital Bank's official policy documents (personal loans, credit cards, complaints).
    Use this for any question about bank rules, fees, rates, limits or procedures. Do not answer such
    questions from memory."""
    docs = policy_retriever.invoke(query)
    if not docs:
        return "No relevant policy text found."
    return "\n\n".join(f"[{d.metadata['source']}, page {d.metadata['page']}]\n{d.page_content}" for d in docs)


print(search_bank_policies.invoke({"query": "foreclosure charge"})[:500])

[Personal_Loan_Policy_2026.pdf, page 2]
5. Fees and charges
All fees below are exclusive of GST at 18%.
Charge
Amount
Processing fee
1.5% of the loan amount, minimum Rs 2,000
Part-prepayment
Free after 6 EMIs have been paid
Foreclosure (full prepayment)
2% of the outstanding principal; not allowed before 6 EMIs
Late payment fee
Rs 500 per missed EMI
Penal charge on overdue amount
2% per month on the overdue EMI amount
Cheque or mandate bounce
Rs 450 per instance
Duplicate statement
Rs 100 per co


In [46]:
bank_agent = create_agent(
    model=llm,
    tools=[search_bank_policies, calculate_emi_tool, get_customer_balance, get_today_date_tool],
    system_prompt=SYSTEM + " For bank rules use search_bank_policies and cite [file, page].",
)

def ask_bank(question):
    out = bank_agent.invoke({"messages": [{"role": "user", "content": question}]})
    used = [c["name"] for m in out["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    print("Q:", question)
    print("tools used:", used)
    print("A:", out["messages"][-1].content, "\n")

ask_bank("What is the foreclosure charge on a personal loan?")
ask_bank("What is my balance?")
ask_bank("My credit score is 750. Search the bank's personal loan rate for that score, then calculate the EMI on Rs 3,00,000 for 24 months.")

Q: What is the foreclosure charge on a personal loan?
tools used: ['search_bank_policies']
A: The foreclosure charge on a personal loan is 2% of the outstanding principal. However, foreclosure is not allowed within the first six months of the loan. After six EMIs have been paid, you may close the loan by paying the outstanding principal, interest up to the date of closure, and the foreclosure charge. [Personal_Loan_Policy_2026.pdf, page 2] 

Q: What is my balance?
tools used: ['get_customer_balance']
A: Your current balance is Rs 48,250.75. 

Q: My credit score is 750. Search the bank's personal loan rate for that score, then calculate the EMI on Rs 3,00,000 for 24 months.
tools used: ['search_bank_policies', 'calculate_emi_tool']
A: For a credit score of 750, the interest rate for a personal loan is 11.75% per year [Personal_Loan_Policy_2026.pdf, page 1].

Regarding your loan of Rs 3,00,000 for 24 months, the details are as follows:
- **EMI:** Rs 14,087.04 per month
- **Total Payable:

The last question needs **both** tools in sequence: search the rate (11.75% for the 740 to 779 band), then calculate with it. Nobody wrote that sequence - the agent planned it from the tool descriptions.

Check the answer yourself: `calculate_emi(300000, 11.75, 24)` gives the exact figure.

In [ ]:
print(calculate_emi(300000, 11.75, 24))

## Part 9: Workflow or agent?

| | Workflow (fixed chain) | Agent (model chooses steps) |
|---|---|---|
| Steps | written by you | decided by the model |
| Predictability | high | lower |
| Cost / latency | fixed, low | variable; often several LLM calls |
| Testing | easy | harder |
| Best for | known, repeatable processes | open-ended requests, many possible paths |

Count the LLM calls an agent made: each `AIMessage` is one model call.

In [ ]:
def count_llm_calls(question):
    out = bank_agent.invoke({"messages": [{"role": "user", "content": question}]})
    from langchain_core.messages import AIMessage
    return sum(isinstance(m, AIMessage) for m in out["messages"])

for q in ["What is the foreclosure charge on a personal loan?",
          "My credit score is 750. Find my loan rate and calculate the EMI on Rs 3,00,000 for 24 months."]:
    print(count_llm_calls(q), "LLM calls |", q)

2 LLM calls | What is the foreclosure charge on a personal loan?


ZeroDivisionError: float division by zero

In [49]:
# blog -> anthropic agents vs workflows
# https://www.anthropic.com/research/building-effective-agents

A RAG chain (Session 4) answers the first question in **one** LLM call. The agent needs at least two (decide to search, then answer). If every question in your product needs a search, a fixed chain is cheaper, faster and easier to test.

**Rule of thumb:** start with the simplest thing that works - a single call, then a chain, then an agent only when the path really varies per request.

## Part 10: Exercises

1. **Easy** - Add a tool `convert_currency(amount, from_currency, to_currency)` using a fixed rate table (USD, EUR, INR). Ask the agent "How much is 250 dollars in rupees?"
2. **Medium** - Add `raise_complaint(category: Literal[...], description: str)` that appends to a Python list and returns a reference number. Make the agent search the grievance policy and tell the user the response time after raising it.
3. **Challenge** - Make the confirm flow work across two turns: keep the `messages` list between calls to `card_agent.invoke`, send "yes" as the second user message, and check that the card is blocked only after confirmation.